In [78]:
import pandas as pd

In [79]:
registry_path = '/Users/jk1/OneDrive - unige.ch/stroke_research/geneva_stroke_unit_dataset/data/stroke_registry/post_hoc_modified/stroke_registry_post_hoc_modified.xlsx'

In [80]:
patient_selection_path = '/Users/jk1/temp/opsum_extration_output/high_frequency_data_patient_selection_with_details.csv'

In [81]:
stroke_df = pd.read_excel(registry_path)

In [82]:
patient_selection_df = pd.read_csv(patient_selection_path, dtype=str)

In [83]:
stroke_df.head()

,Unnamed: 0,Case ID,Centre,Entry date,Entry person,Patient refuses use of data for research,Last name,First name,DOB,Type of event,...,3M NIHSS,3M Stroke,3M Stroke date,3M ICH,3M ICH date,3M Death,3M Death date,3M Death cause,3M Epileptic seizure,3M Epileptic seizure date
0,1,SSR-HUG-023804235,HUG Genève,20180525,Emmanuel Carrera,yes,NaN,NaN,19310702.0,Ischemic stroke,...,NaN,no,NaN,no,NaN,yes,20180731.0,other vascular death,no,NaN
1,2,SSR-HUG-1002048465,HUG Genève,20181029,Emmanuel Carrera,NaN,NaN,NaN,19471020.0,Transient ischemic attack,...,0.0,no,NaN,no,NaN,no,NaN,NaN,no,NaN
2,3,SSR-HUG-1002910806,HUG Genève,20181123,Emmanuel Carrera,NaN,NaN,NaN,19290317.0,Transient ischemic attack,...,NaN,no,NaN,no,NaN,no,NaN,NaN,no,NaN
3,4,SSR-HUG-1005030884,HUG Genève,20181108,Emmanuel Carrera,NaN,NaN,NaN,19331031.0,Ischemic stroke,...,2.0,no,NaN,no,NaN,no,NaN,NaN,NaN,NaN
4,5,SSR-HUG-1005303803,HUG Genève,20180922,Emmanuel Carrera,NaN,NaN,NaN,19451125.0,Intracerebral hemorrhage,...,NaN,no,NaN,no,NaN,no,NaN,NaN,no,NaN


In [84]:
patient_selection_df.head()

,Unnamed: 0,patient_id,EDS_last_4_digits,manual_eds,manual_patient_id,DOB,Arrival at hospital,Arrival time,Discharge date,Discharge time,Death at hospital date,Death at hospital time,Time of symptom onset known,Stroke onset date,Stroke onset time,Referral
0,0,100503,0884,NaN,NaN,19331031.0,20181108,13:21,20181116.0,13:00,NaN,NaN,no,20181108.0,08:30,Emergency service (144)
1,1,1005564,4109,NaN,NaN,19570311.0,20181002,08:11,20181022.0,09:35,NaN,NaN,no,20180928.0,00:00,Emergency service (144)
2,2,1005798,9217,NaN,NaN,19500315.0,20180910,21:36,20180918.0,17:47,NaN,NaN,yes,20180908.0,09:00,Other hospital
3,3,1011794,0030,NaN,NaN,19210723.0,20180122,13:13,NaN,NaN,20180201.0,01:00,yes,20180122.0,11:30,Emergency service (144)
4,4,1012915,7747,NaN,NaN,19350829.0,20180520,19:07,20180524.0,12:37,NaN,NaN,yes,20180520.0,16:00,Emergency service (144)


In [85]:
from preprocessing.utils import create_registry_case_identification_column

patient_selection_df['case_admission_id'] = create_registry_case_identification_column(patient_selection_df)
stroke_df['case_admission_id'] = create_registry_case_identification_column(stroke_df)

In [86]:
restricted_stroke_df = stroke_df[stroke_df.case_admission_id.isin(patient_selection_df.case_admission_id)]

In [87]:
# if death in hospital, set mRs to 6
restricted_stroke_df.loc[restricted_stroke_df['Death in hospital'] == 'yes', '3M mRS'] = 6
# if 3M Death and 3M mRS nan, set mrs to 6
restricted_stroke_df.loc[(restricted_stroke_df['3M Death'] == 'yes') &
                                    (restricted_stroke_df['3M mRS'].isna()), '3M mRS'] = 6

/Users/jk1/opt/anaconda3/envs/opsum/lib/python3.8/site-packages/pandas/core/indexing.py:1817: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_column(loc, value, pi)


In [88]:
n_total_missing_outcomes =  restricted_stroke_df['3M mRS'].isna().sum()
print(f'Number of total missing outcomes: {n_total_missing_outcomes}, {n_total_missing_outcomes/len(restricted_stroke_df)}; n=  {len(restricted_stroke_df)}')

Number of total missing outcomes: 399, 0.12058023572076156; n=  3309


In [89]:
# find cids from transfers from France (where Non-Swiss == yes & referral == other hospital)
cids_transfers_from_france = restricted_stroke_df[(restricted_stroke_df['Referral'] == 'Other hospital') & (restricted_stroke_df['Non-Swiss'] == 'yes')]['case_admission_id'].values

In [90]:
n_missing_outcomes_transfers_from_france = restricted_stroke_df[restricted_stroke_df.case_admission_id.isin(cids_transfers_from_france)]['3M mRS'].isna().sum()
print(f'Number of missing outcomes in transfers from france:', n_missing_outcomes_transfers_from_france, 'for a number of transfers from France:', len(cids_transfers_from_france))

Number of missing outcomes in transfers from france: 118 for a number of transfers from France: 335


In [ ]:
n_missing_outcomes_for_non_swiss_residents = restricted_stroke_df[restricted_stroke_df['Non-Swiss'] == 'yes']['3M mRS'].isna().sum()
print(f'Number of missing outcomes for non swiss residents:', n_missing_outcomes_for_non_swiss_residents, 'of:', len(restricted_stroke_df[restricted_stroke_df['Non-Swiss'] == 'yes']))

In [ ]:
n_missing_outcomes_all_transfers = restricted_stroke_df[(restricted_stroke_df['Referral'] == 'Other hospital')]['3M mRS'].isna().sum()
print(f'Number of missing outcomes for all transferred patients:', n_missing_outcomes_all_transfers, f'for a total of {len(restricted_stroke_df[restricted_stroke_df["Referral"] == "Other hospital"])} events')

In [ ]:
n_missing_outcomes_in_hospital = restricted_stroke_df[restricted_stroke_df['Referral'] == 'In-hospital event']['3M mRS'].isna().sum()
print(f'Number of missing outcomes for in-hospital events:', n_missing_outcomes_in_hospital, f'for a total of {len(restricted_stroke_df[restricted_stroke_df["Referral"] == "In-hospital event"])} events')

In [ ]:
restricted_stroke_df[(restricted_stroke_df['Non-Swiss'] != 'yes') & (restricted_stroke_df['Referral'] != 'In-hospital event') & (restricted_stroke_df['3M mRS'].isna())]